In [ ]:
# Feature Engineering

The raw-feature boosting models appear to be approaching a performance
ceiling around 0.942 OOF ROC-AUC.

This notebook tests whether explicitly constructed interaction features
can expose additional predictive structure.

Initial features are based only on existing input columns and do not
use the target, so they are safe from target leakage.

In [1]:
import pandas as pd
import numpy as np

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train.shape, test.shape

((668665, 15), (286571, 14))

In [3]:
def add_features(df):
    df = df.copy()

    # --------------------------------------------------
    # Numerical interactions
    # --------------------------------------------------

    df["Income_per_Car"] = (
        df["Annual_Income_USD"]
        / df["Number_of_Cars_Owned"].clip(lower=1)
    )

    df["Total_Charging_Stations"] = (
        df["Charging_Stations_Near_Home"]
        + df["Charging_Stations_Near_Work"]
    )

    df["Charging_Station_Difference"] = (
        df["Charging_Stations_Near_Home"]
        - df["Charging_Stations_Near_Work"]
    )

    df["Income_x_Concern"] = (
        df["Annual_Income_USD"]
        * df["Environmental_Concern_Level"]
    )

    df["Commute_x_Concern"] = (
        df["Daily_Commute_km"]
        * df["Environmental_Concern_Level"]
    )

    # --------------------------------------------------
    # Categorical interactions
    # --------------------------------------------------

    df["Subsidy_x_Concern"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Environmental_Concern_Level"].astype(str)
    )

    df["Subsidy_x_Anxiety"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    df["Concern_x_Anxiety"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    df["Income_x_Subsidy"] = (
        pd.cut(
            df["Annual_Income_USD"],
            bins=[
                -np.inf,
                60000,
                80000,
                100000,
                120000,
                np.inf
            ],
            labels=[
                "Income_1",
                "Income_2",
                "Income_3",
                "Income_4",
                "Income_5"
            ]
        ).astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    df["HomeCharge_x_Subsidy"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    df["HomeCharge_x_Commute"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + pd.cut(
            df["Daily_Commute_km"],
            bins=[
                -np.inf,
                20,
                40,
                60,
                np.inf
            ],
            labels=[
                "Short",
                "Medium",
                "Long",
                "VeryLong"
            ]
        ).astype(str)
    )

    return df

In [4]:
train_fe = add_features(train)
test_fe = add_features(test)

train.shape, train_fe.shape

((668665, 15), (668665, 26))

In [5]:
new_features = [
    col
    for col in train_fe.columns
    if col not in train.columns
]

new_features

['Income_per_Car',
 'Total_Charging_Stations',
 'Charging_Station_Difference',
 'Income_x_Concern',
 'Commute_x_Concern',
 'Subsidy_x_Concern',
 'Subsidy_x_Anxiety',
 'Concern_x_Anxiety',
 'Income_x_Subsidy',
 'HomeCharge_x_Subsidy',
 'HomeCharge_x_Commute']

In [6]:
train_fe[new_features].head()

,Income_per_Car,Total_Charging_Stations,Charging_Station_Difference,Income_x_Concern,Commute_x_Concern,Subsidy_x_Concern,Subsidy_x_Anxiety,Concern_x_Anxiety,Income_x_Subsidy,HomeCharge_x_Subsidy,HomeCharge_x_Commute
0,46443.5,10,-4,92887.0,23.4,No_1.0,No_Low,1.0_Low,Income_3_No,Yes_No,Yes_Medium
1,30000.0,4,0,120000.0,20.0,No_4.0,No_Low,4.0_Low,Income_1_No,Yes_No,Yes_Short
2,94389.0,23,-7,471945.0,184.0,Yes_5.0,Yes_Low,5.0_Low,Income_3_Yes,No_Yes,No_Medium
3,36790.0,15,-3,220740.0,71.1,No_3.0,No_Low,3.0_Low,Income_2_No,Yes_No,Yes_Medium
4,57898.0,5,-1,173694.0,152.4,No_3.0,No_Low,3.0_Low,Income_1_No,Yes_No,Yes_Long


In [7]:
y = train_fe["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

In [8]:
X_fe = train_fe.drop(
    columns=["id", "Will_Buy_EV"]
)

X_test_fe = test_fe.drop(
    columns=["id"]
)

In [9]:
categorical_features_fe = X_fe.select_dtypes(
    include=["object", "category"]
).columns.tolist()

categorical_features_fe

/tmp/ipykernel_12738/962381165.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_fe = X_fe.select_dtypes(


['Gender',
 'City_Type',
 'Current_Car_Type',
 'Home_Charging_Possible',
 'Subsidy_Available',
 'Range_Anxiety_Level',
 'Subsidy_x_Concern',
 'Subsidy_x_Anxiety',
 'Concern_x_Anxiety',
 'Income_x_Subsidy',
 'HomeCharge_x_Subsidy',
 'HomeCharge_x_Commute']

In [10]:
for col in categorical_features_fe:
    X_fe[col] = X_fe[col].astype("category")
    X_test_fe[col] = X_test_fe[col].astype("category")

In [11]:
X_fe.shape, X_test_fe.shape

((668665, 24), (286571, 24))

In [12]:
X_train_fe, X_valid_fe, y_train_fe, y_valid_fe = train_test_split(
    X_fe,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [19]:
lgb_fe_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [20]:
lgb_fe_model.fit(
    X_train_fe,
    y_train_fe,

    eval_X=X_valid_fe,
    eval_y=y_valid_fe,

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.940913	valid_0's binary_logloss: 0.228191
[400]	valid_0's auc: 0.941515	valid_0's binary_logloss: 0.227038
[600]	valid_0's auc: 0.941674	valid_0's binary_logloss: 0.226732
[800]	valid_0's auc: 0.941749	valid_0's binary_logloss: 0.226585
[1000]	valid_0's auc: 0.941772	valid_0's binary_logloss: 0.226533
Early stopping, best iteration is:
[972]	valid_0's auc: 0.941785	valid_0's binary_logloss: 0.226516
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [21]:
fe_valid_pred = lgb_fe_model.predict_proba(
    X_valid_fe
)[:, 1]

fe_auc = roc_auc_score(
    y_valid_fe,
    fe_valid_pred
)

fe_auc

0.9417849192450622

In [22]:
feature_importance_fe = pd.DataFrame({
    "Feature": X_fe.columns,
    "Importance": lgb_fe_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

feature_importance_fe.head(25)

,Feature,Importance
1,Annual_Income_USD,5042
16,Income_x_Concern,3172
2,Daily_Commute_km,3075
0,Age,3072
13,Income_per_Car,2852
17,Commute_x_Concern,2007
14,Total_Charging_Stations,1575
15,Charging_Station_Difference,1323
5,Charging_Stations_Near_Work,1175
4,Charging_Stations_Near_Home,1125


In [23]:
feature_importance_fe[
    feature_importance_fe["Feature"].isin(new_features)
]

,Feature,Importance
16,Income_x_Concern,3172
13,Income_per_Car,2852
17,Commute_x_Concern,2007
14,Total_Charging_Stations,1575
15,Charging_Station_Difference,1323
18,Subsidy_x_Concern,701
20,Concern_x_Anxiety,693
23,HomeCharge_x_Commute,642
21,Income_x_Subsidy,567
19,Subsidy_x_Anxiety,171


In [24]:
print(f"Raw LightGBM AUC:        {0.941669:.6f}")
print(f"Feature Engineered AUC: {fe_auc:.6f}")
print(f"Difference:             {fe_auc - 0.941669:+.6f}")

Raw LightGBM AUC:        0.941669
Feature Engineered AUC: 0.941785
Difference:             +0.000116


In [25]:
from sklearn.model_selection import StratifiedKFold

In [26]:
cv_fe = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions_fe = np.zeros(len(X_fe))
test_predictions_fe = np.zeros(len(X_test_fe))

fold_scores_fe = []
best_iterations_fe = []

In [27]:
for fold, (train_idx, valid_idx) in enumerate(
    cv_fe.split(X_fe, y),
    start=1
):

    print(f"\n{'=' * 50}")
    print(f"FOLD {fold}")
    print(f"{'=' * 50}")

    X_train_fold = X_fe.iloc[train_idx]
    X_valid_fold = X_fe.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    model_fe = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,

        subsample=0.9,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        random_state=42 + fold,
        n_jobs=-1,
        verbosity=-1
    )

    model_fe.fit(
        X_train_fold,
        y_train_fold,

        eval_X=X_valid_fold,
        eval_y=y_valid_fold,

        eval_metric="auc",

        callbacks=[
            early_stopping(
                150,
                first_metric_only=True
            ),
            log_evaluation(0)
        ]
    )

    valid_pred = model_fe.predict_proba(
        X_valid_fold
    )[:, 1]

    test_pred = model_fe.predict_proba(
        X_test_fe
    )[:, 1]

    oof_predictions_fe[valid_idx] = valid_pred
    test_predictions_fe += test_pred / 5

    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_pred
    )

    fold_scores_fe.append(fold_auc)
    best_iterations_fe.append(
        model_fe.best_iteration_
    )

    print(f"Fold {fold} AUC: {fold_auc:.6f}")
    print(f"Best iteration: {model_fe.best_iteration_}")


FOLD 1
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1001]	valid_0's auc: 0.940593	valid_0's binary_logloss: 0.228609
Evaluated only: auc
Fold 1 AUC: 0.940593
Best iteration: 1001

FOLD 2
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1005]	valid_0's auc: 0.941624	valid_0's binary_logloss: 0.226866
Evaluated only: auc
Fold 2 AUC: 0.941624
Best iteration: 1005

FOLD 3
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1187]	valid_0's auc: 0.94296	valid_0's binary_logloss: 0.224567
Evaluated only: auc
Fold 3 AUC: 0.942960
Best iteration: 1187

FOLD 4
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[981]	valid_0's auc: 0.942345	valid_0's binary_logloss: 0.225526
Evaluated only: auc
Fold 4 AUC: 0.942345
Best iteration: 981

FOLD 5
Training until validation scores don't improve for 150 rounds
E

In [28]:
oof_auc_fe = roc_auc_score(
    y,
    oof_predictions_fe
)

print("\nFold scores:")

for fold, score in enumerate(
    fold_scores_fe,
    start=1
):
    print(f"Fold {fold}: {score:.6f}")

print()
print(f"Mean fold AUC: {np.mean(fold_scores_fe):.6f}")
print(f"OOF AUC:       {oof_auc_fe:.6f}")
print(f"Std AUC:       {np.std(fold_scores_fe):.6f}")

print()
print("Best iterations:")
print(best_iterations_fe)


Fold scores:
Fold 1: 0.940593
Fold 2: 0.941624
Fold 3: 0.942960
Fold 4: 0.942345
Fold 5: 0.941878

Mean fold AUC: 0.941880
OOF AUC:       0.941871
Std AUC:       0.000788

Best iterations:
[1001, 1005, 1187, 981, 945]


In [29]:
fe_oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "lgb_fe_oof": oof_predictions_fe
})

fe_oof_df.to_csv(
    "../outputs/predictions/lightgbm_feature_engineered_oof.csv",
    index=False
)

In [30]:
fe_test_df = pd.DataFrame({
    "id": test["id"],
    "lgb_fe_prediction": test_predictions_fe
})

fe_test_df.to_csv(
    "../outputs/predictions/lightgbm_feature_engineered_test.csv",
    index=False
)

In [31]:
cat_oof = pd.read_csv(
    "../outputs/predictions/catboost_oof.csv"
)

lgb_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_oof.csv"
)

xgb_oof = pd.read_csv(
    "../outputs/predictions/xgboost_oof.csv"
)

fe_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_feature_engineered_oof.csv"
)

In [32]:
assert cat_oof["id"].equals(lgb_oof["id"])
assert cat_oof["id"].equals(xgb_oof["id"])
assert cat_oof["id"].equals(fe_oof["id"])

assert cat_oof["target"].equals(lgb_oof["target"])
assert cat_oof["target"].equals(xgb_oof["target"])
assert cat_oof["target"].equals(fe_oof["target"])

print("All four OOF files aligned.")

All four OOF files aligned.


In [33]:
prediction_matrix_4 = pd.DataFrame({
    "CatBoost": cat_oof["catboost_oof"],
    "LightGBM": lgb_oof["lightgbm_oof"],
    "XGBoost": xgb_oof["xgboost_oof"],
    "LightGBM_FE": fe_oof["lgb_fe_oof"]
})

prediction_matrix_4.corr()

,CatBoost,LightGBM,XGBoost,LightGBM_FE
CatBoost,1.000000,0.997437,0.997973,0.997185
LightGBM,0.997437,1.000000,0.997616,0.998324
XGBoost,0.997973,0.997616,1.000000,0.997179
LightGBM_FE,0.997185,0.998324,0.997179,1.000000


In [34]:
y_ensemble = cat_oof["target"]

cat_rank = cat_oof["catboost_oof"].rank(
    method="average",
    pct=True
)

lgb_rank = lgb_oof["lightgbm_oof"].rank(
    method="average",
    pct=True
)

xgb_rank = xgb_oof["xgboost_oof"].rank(
    method="average",
    pct=True
)

fe_rank = fe_oof["lgb_fe_oof"].rank(
    method="average",
    pct=True
)

In [35]:
current_best_rank = (
    0.20 * cat_rank
    + 0.35 * lgb_rank
    + 0.45 * xgb_rank
)

current_best_auc = roc_auc_score(
    y_ensemble,
    current_best_rank
)

current_best_auc

0.9419881806955798

In [36]:
fe_replacement_rank = (
    0.20 * cat_rank
    + 0.35 * fe_rank
    + 0.45 * xgb_rank
)

fe_replacement_auc = roc_auc_score(
    y_ensemble,
    fe_replacement_rank
)

fe_replacement_auc

0.9420459541460551

In [37]:
lgb_family_average = (
    0.20 * cat_rank
    + 0.175 * lgb_rank
    + 0.175 * fe_rank
    + 0.45 * xgb_rank
)

lgb_family_average_auc = roc_auc_score(
    y_ensemble,
    lgb_family_average
)

lgb_family_average_auc

0.9420304008831537

In [38]:
family_results = []

for fe_share in np.arange(0.0, 1.01, 0.05):

    lgb_family = (
        (1 - fe_share) * lgb_rank
        + fe_share * fe_rank
    )

    blended = (
        0.20 * cat_rank
        + 0.35 * lgb_family
        + 0.45 * xgb_rank
    )

    auc = roc_auc_score(
        y_ensemble,
        blended
    )

    family_results.append({
        "FE_Share_of_LGB_Weight": fe_share,
        "Raw_LGB_Share": 1 - fe_share,
        "OOF_AUC": auc
    })

family_results = pd.DataFrame(family_results)

family_results.sort_values(
    "OOF_AUC",
    ascending=False
).head(10)

,FE_Share_of_LGB_Weight,Raw_LGB_Share,OOF_AUC
20,1.00,0.00,0.942046
19,0.95,0.05,0.942046
18,0.90,0.10,0.942045
17,0.85,0.15,0.942044
16,0.80,0.20,0.942043
15,0.75,0.25,0.942042
14,0.70,0.30,0.942040
13,0.65,0.35,0.942038
12,0.60,0.40,0.942036
11,0.55,0.45,0.942033


In [39]:
best_family_blend = (
    family_results
    .sort_values("OOF_AUC", ascending=False)
    .iloc[0]
)

best_family_blend

FE_Share_of_LGB_Weight    1.000000
Raw_LGB_Share             0.000000
OOF_AUC                   0.942046
Name: 20, dtype: float64

In [40]:
fe_add_results = []

for fe_weight in np.arange(0.0, 0.51, 0.05):

    remaining = 1 - fe_weight

    blended = (
        remaining * current_best_rank
        + fe_weight * fe_rank
    )

    auc = roc_auc_score(
        y_ensemble,
        blended
    )

    fe_add_results.append({
        "FE_Weight": fe_weight,
        "Existing_Ensemble_Weight": remaining,
        "OOF_AUC": auc
    })

fe_add_results = pd.DataFrame(fe_add_results)

fe_add_results.sort_values(
    "OOF_AUC",
    ascending=False
)

,FE_Weight,Existing_Ensemble_Weight,OOF_AUC
7,0.35,0.65,0.942045
8,0.40,0.60,0.942045
6,0.30,0.70,0.942044
9,0.45,0.55,0.942042
5,0.25,0.75,0.942040
10,0.50,0.50,0.942037
4,0.20,0.80,0.942034
3,0.15,0.85,0.942026
2,0.10,0.90,0.942015
1,0.05,0.95,0.942003


In [41]:
best_fe_addition = (
    fe_add_results
    .sort_values("OOF_AUC", ascending=False)
    .iloc[0]
)

best_fe_addition

FE_Weight                   0.350000
Existing_Ensemble_Weight    0.650000
OOF_AUC                     0.942045
Name: 7, dtype: float64

In [42]:
print(f"Raw LightGBM:             {0.941770:.6f}")
print(f"FE LightGBM:              {oof_auc_fe:.6f}")
print(f"Current 3-model ensemble: {current_best_auc:.6f}")
print(f"FE replaces LightGBM:     {fe_replacement_auc:.6f}")
print(f"50/50 LGB family:         {lgb_family_average_auc:.6f}")
print()
print("Best LGB-family blend:")
print(best_family_blend)
print()
print("Best FE addition:")
print(best_fe_addition)

Raw LightGBM:             0.941770
FE LightGBM:              0.941871
Current 3-model ensemble: 0.941988
FE replaces LightGBM:     0.942046
50/50 LGB family:         0.942030

Best LGB-family blend:
FE_Share_of_LGB_Weight    1.000000
Raw_LGB_Share             0.000000
OOF_AUC                   0.942046
Name: 20, dtype: float64

Best FE addition:
FE_Weight                   0.350000
Existing_Ensemble_Weight    0.650000
OOF_AUC                     0.942045
Name: 7, dtype: float64
